# Faster R-CNN -- Sofia domain-adaptation fine-tuning (notebook 12)

Direct counterpart to notebook 11 (YOLOv8n Sofia fine-tuning). Starts from
the RDD2022-large-trained Faster R-CNN (notebook 10, mAP50-95 0.186 on the
RDD2022 test set) and fine-tunes it on the same 131 manually collected Sofia
photos, evaluated against the exact same 42-image held-out Sofia test set
YOLO's notebook 11 used -- so the two architectures' fine-tuning gains are
directly comparable.

## Data (same split as notebook 11)

| Split | Images | Role |
|---|---|---|
| fine-tune train | 131 | trains the fine-tuning stage |
| fine-tune val | 28 | monitors fine-tuning for overfitting |
| **held-out test** | **42** | **never trained on** -- before/after comparison only |

`src/training/fasterrcnn.py`'s `build_dataloaders()` assumes a single dataset
directory with train/val/test as sibling folders, which doesn't match this
split (train/val live under `data/manual_finetune_split/`, test lives under
`data/manual_split/test/`) -- so this notebook builds the three
`YoloDetectionDataset` loaders directly instead, reusing the same underlying
dataset class.

## Why a low learning rate, short run

Same reasoning as notebook 11: 131 images is tiny relative to the
4,291-image RDD2022-large run, so this is meant to nudge the already-trained
weights toward Sofia's conditions, not retrain from scratch. Reduced
learning rate (0.0005 vs. the from-scratch run's 0.005) and aggressive early
stopping (patience=8).

In [1]:
import sys
from pathlib import Path

import torch
import pandas as pd
from torch.utils.data import DataLoader

sys.path.append("..")
from src.data.yolo_torchvision_dataset import YoloDetectionDataset, collate_fn
from src.training.fasterrcnn import build_model, TrainConfig, get_device, fit, evaluate, CLASS_NAMES

torch.manual_seed(42)

## Data loaders

Train/val from the fine-tune split, test from the fully held-out Sofia split.

In [2]:
FINETUNE_DIR = Path("../data/manual_finetune_split")
TEST_DIR = Path("../data/manual_split/test")
LARGE_MODEL_PATH = Path("../models/fasterrcnn_large/best.pt")

BATCH_SIZE = 4
WORKERS = 0

train_dataset = YoloDetectionDataset(FINETUNE_DIR / "train" / "images", FINETUNE_DIR / "train" / "labels")
val_dataset = YoloDetectionDataset(FINETUNE_DIR / "val" / "images", FINETUNE_DIR / "val" / "labels")
test_dataset = YoloDetectionDataset(TEST_DIR / "images", TEST_DIR / "labels")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=WORKERS, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=WORKERS, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=WORKERS, collate_fn=collate_fn)

print(f"fine-tune train: {len(train_dataset)} images ({len(train_loader)} batches)")
print(f"fine-tune val:   {len(val_dataset)} images ({len(val_loader)} batches)")
print(f"held-out test:   {len(test_dataset)} images ({len(test_loader)} batches)")

fine-tune train: 131 images (33 batches)
fine-tune val:   28 images (7 batches)
held-out test:   42 images (11 batches)


## Step 1 -- baseline: evaluate the RDD2022-large model on the held-out Sofia test set (no fine-tuning yet)

In [3]:
DEVICE = get_device()
print("Device:", DEVICE)

baseline_model = build_model(imgsz=640)
baseline_model.load_state_dict(torch.load(LARGE_MODEL_PATH, map_location=DEVICE))
baseline_model.to(DEVICE)

baseline_results = evaluate(baseline_model, test_loader, DEVICE, iou_threshold=0.5)

print("Baseline (no fine-tuning) on Sofia held-out test set:")
print(f"  Precision: {baseline_results['precision']:.3f}  (at confidence >= {baseline_results['best_threshold']})")
print(f"  Recall:    {baseline_results['recall']:.3f}")
print(f"  F1:        {baseline_results['f1']:.3f}")
print(f"  mAP50:     {baseline_results['mAP50']:.4f}")
print(f"  mAP50-95:  {baseline_results['mAP50-95']:.4f}")

Device: cuda
Baseline (no fine-tuning) on Sofia held-out test set:
  Precision: 0.308  (at confidence >= 0.45)
  Recall:    0.169
  F1:        0.218
  mAP50:     0.1065
  mAP50-95:  0.0397


## Step 2 -- fine-tune on the Sofia fine-tune split

In [4]:
finetune_model = build_model(imgsz=640)
finetune_model.load_state_dict(torch.load(LARGE_MODEL_PATH, map_location=DEVICE))
finetune_model.to(DEVICE)

cfg = TrainConfig(
    epochs=30,
    patience=8,
    learning_rate=0.0005,  # reduced from the from-scratch run's 0.005 -- nudge, not retrain
    momentum=0.9,
    weight_decay=0.0005,
    step_size=15,
    gamma=0.1,
    accum_steps=4,   # 4 * 4 = effective batch 16, matches the from-scratch runs
    amp=True,
)
print(cfg)

RUN_DIR = Path("../models/fasterrcnn_sofia_finetune")

history_df = fit(finetune_model, train_loader, val_loader, DEVICE, RUN_DIR, cfg=cfg, progress=True)
print(f"Epochs run: {len(history_df)} (max was {cfg.epochs}, patience {cfg.patience})")
history_df

TrainConfig(epochs=30, patience=8, learning_rate=0.0005, momentum=0.9, weight_decay=0.0005, step_size=15, gamma=0.1, accum_steps=4, amp=True, grad_clip=5.0)


epochs:   0%|          | 0/30 [00:00<?, ?epoch/s]

Early stopping: no val_loss improvement for 8 epochs (epoch 28).
Epochs run: 28 (max was 30, patience 8)


,epoch,train_loss,val_loss,time_s
0,1,1.269662,1.007155,54.183043
1,2,1.102538,0.904291,44.988377
2,3,0.904929,0.807473,53.186556
3,4,0.759826,0.755374,45.881374
4,5,0.685925,0.723138,41.831813
5,6,0.650377,0.681280,41.552011
6,7,0.607828,0.660183,40.851665
7,8,0.591822,0.657879,19.302161
8,9,0.560539,0.640282,20.016086
9,10,0.542167,0.639410,19.505164


## Step 3 -- evaluate the fine-tuned model on the same held-out Sofia test set

In [5]:
finetune_model.load_state_dict(torch.load(RUN_DIR / "best.pt", map_location=DEVICE))

finetuned_results = evaluate(finetune_model, test_loader, DEVICE, iou_threshold=0.5)

print("Fine-tuned model on Sofia held-out test set:")
print(f"  Precision: {finetuned_results['precision']:.3f}  (at confidence >= {finetuned_results['best_threshold']})")
print(f"  Recall:    {finetuned_results['recall']:.3f}")
print(f"  F1:        {finetuned_results['f1']:.3f}")
print(f"  mAP50:     {finetuned_results['mAP50']:.4f}")
print(f"  mAP50-95:  {finetuned_results['mAP50-95']:.4f}")

Fine-tuned model on Sofia held-out test set:
  Precision: 0.588  (at confidence >= 0.6)
  Recall:    0.211
  F1:        0.311
  mAP50:     0.1490
  mAP50-95:  0.0605


## Before / after comparison -- did domain-adaptation fine-tuning help?

In [6]:
comparison_df = pd.DataFrame([
    {
        "model": "Faster R-CNN RDD2022-large only (no fine-tuning)",
        "precision": round(baseline_results["precision"], 3),
        "recall": round(baseline_results["recall"], 3),
        "f1": round(baseline_results["f1"], 3),
        "mAP50": round(baseline_results["mAP50"], 4),
        "mAP50-95": round(baseline_results["mAP50-95"], 4),
    },
    {
        "model": "Faster R-CNN RDD2022-large + Sofia fine-tuned",
        "precision": round(finetuned_results["precision"], 3),
        "recall": round(finetuned_results["recall"], 3),
        "f1": round(finetuned_results["f1"], 3),
        "mAP50": round(finetuned_results["mAP50"], 4),
        "mAP50-95": round(finetuned_results["mAP50-95"], 4),
    },
])

map50_95_delta = finetuned_results["mAP50-95"] - baseline_results["mAP50-95"]
print(f"mAP50-95 change from fine-tuning: {map50_95_delta:+.4f} "
      f"({'improved' if map50_95_delta > 0 else 'did not improve'})")

comparison_df

mAP50-95 change from fine-tuning: +0.0208 (improved)


,model,precision,recall,f1,mAP50,mAP50-95
0,Faster R-CNN RDD2022-large only (no fine-tuning),0.308,0.169,0.218,0.1065,0.0397
1,Faster R-CNN RDD2022-large + Sofia fine-tuned,0.588,0.211,0.311,0.1490,0.0605


## Notes

- Same 42-image held-out Sofia test set as notebook 11 -- directly comparable
  before/after deltas across both architectures.
- Combine this notebook's and notebook 11's before/after deltas into one
  final table for the thesis: does domain-adaptation fine-tuning help more
  for one architecture than the other, and does either approach the other's
  from-scratch RDD2022-large performance once adapted to Sofia conditions?